# 04 · Seeing the reaction of the brain

**Goal:** feed the MaleCNS brain a looming signal and watch it respond over time, then ask *what makes it respond*.

You will learn
- the brain's update rule (`fly/brain.py`): a **leaky rate model** on the real wiring matrix **W**
- how input goes in (`brain.drive`) and how output comes out (`brain.rates`, per group)
- how a group's rate becomes a button (`actions/jump.py`)
- how to run small experiments on the brain: different inputs, and **removing the recurrence**

> Real vs invented: **W** (who connects to whom, with signs) is **teal**. The update rule's four numbers, the input levels and the jump thresholds are **orange**: we chose them.

## 0. Data

Set the folder with the MaleCNS files (see `tutorials/README.md`). The cell after builds the brain, or loads it from the cache if notebook 02 (or `build-brains`) already made it.

In [ ]:
DATA_DIR = "~/fly_mario_fly_data"      # <-- change this to the folder that holds malecns_v1.0/

import numpy as np
import matplotlib.pyplot as plt

import _helpers as H
H.use_data_folder(DATA_DIR)

from fly_mario_fly import registry
from fly_mario_fly.fly import brain as fb
from fly_mario_fly.runner import load_fly, brain_spec
from fly_mario_fly.senses.kit.geom import theta_of

In [ ]:
fly = load_fly("looming_only")                 # a fly config from configs/flies/: only the looming sense is attached
brain = fb.build_brain(brain_spec(fly))
print("neurons:", brain.n)
print("groups :", brain.counts())

## 1. The update rule

Each neuron has a **rate** *h* (think: how active it is, never below 0). Per game frame, the brain runs 10 small steps of:

$$h \leftarrow (1-\alpha)\,h + \alpha\,\mathrm{clip}\big(g\,W h + I,\ 0,\ \text{clip}\big)$$

- **W h**: every neuron listens to the others through the real wiring (excitatory or inhibitory),
- **I**: the input from the senses (zero for most neurons),
- **α**: how fast activity follows its drive ("leaky"), **g**: overall coupling, **clip**: a ceiling.

It is the code in `fb.step`. Note the docstring of `RateModel`: *"All four numbers are hand-set (invented), not read off the connectome."*

In [ ]:
model = fb.RateModel()
print(model)

## 2. Poke the brain: a constant input

`brain.drive({"LPLC2_R": level})` builds an input vector with that level on every neuron of the group. `brain.advance(h, I)` runs one game frame; `brain.rates(h)` gives the mean rate of every group.
Let's turn on the right looming detectors at a fixed strength and watch.

In [ ]:
def simulate(inputs_at, frames, brain=brain):
    """inputs_at(t) -> {group: level} for frame t. Returns {group: array of mean rates over time}."""
    h = brain.new_state()
    out = []
    for t in range(frames):
        h = brain.advance(h, brain.drive(inputs_at(t)), model)
        out.append(brain.rates(h))
    return {g: np.array([o[g] for o in out]) for g in out[0]}

r = simulate(lambda t: {"LPLC2_R": 0.5}, 60)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for g in ["LPLC2_R", "LPLC2_L", "LC4_R", "DNp01", "TTMn"]:
    ax.plot(r[g], label=g)
ax.set_xlabel("game frame"); ax.set_ylabel("mean rate"); ax.legend(); ax.set_title("a constant looming input to the right LPLC2 neurons")
plt.show()
print("final rates:", {g: round(float(v[-1]), 3) for g, v in r.items() if v[-1] > 0.005})
print("LPLC2_L (left eye, not driven) ends at", round(float(r["LPLC2_L"][-1]), 4))

Things to notice: LPLC2_R rises towards the input; **DNp01** (the Giant Fiber) and then **TTMn** (the jump motor neuron) follow, delayed and smaller, because they get their drive only through the wiring. The *left* LPLC2 neurons stay at exactly 0 (nothing connects the two eyes' detectors), the walking neuron DNp09 picks up a little, and the LC10a steering cells hardly react. 

## 3. How strong an input does it take?

In [ ]:
levels = [0.1, 0.3, 0.6, 1.0, 1.5]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
peaks = []
for L in levels:
    r = simulate(lambda t: {"LPLC2_R": L}, 80)
    ax.plot(r["TTMn"], label=f"input {L}")
    peaks.append(float(r["TTMn"].max()))
ax.set_xlabel("game frame"); ax.set_ylabel("TTMn rate"); ax.legend(); ax.set_title("the jump motor neuron for different input strengths")
plt.show()
print("peak TTMn / input:", [round(p / L, 3) for p, L in zip(peaks, levels)])

The ratios are almost equal: in this regime the response is **close to proportional** to the input (the clip never bites). The non-linearity in behaviour will come from the *threshold* in the action below, not from the brain.

## 4. A goomba approaches

Now a realistic input. Use the looming angle from notebook 03: θ for LPLC2 and |Δθ| for LC4, as a goomba closes in from 150 px to 4 px at 1.5 px per frame.

In [ ]:
gaps = np.arange(150, 4, -1.5)
theta = np.array([theta_of(g, 16) for g in gaps])
dtheta = np.abs(np.diff(theta, prepend=theta[0]))
r = simulate(lambda t: {"LPLC2_R": theta[t], "LC4_R": dtheta[t]}, len(gaps))

fig, axes = plt.subplots(4, 1, figsize=(7.5, 8.5), sharex=True)
axes[0].plot(gaps, color="grey"); axes[0].set_ylabel("gap (px)")
axes[1].plot(theta, label="theta -> LPLC2_R"); axes[1].plot(dtheta * 10, label="10 x |dtheta| -> LC4_R"); axes[1].legend(); axes[1].set_ylabel("input")
axes[2].plot(r["LPLC2_R"], label="LPLC2_R"); axes[2].plot(r["LC4_R"], label="LC4_R"); axes[2].plot(r["DNp01"], label="DNp01 (Giant Fiber)"); axes[2].legend(); axes[2].set_ylabel("rate")
axes[3].plot(r["TTMn"], color="tab:red", label="TTMn")
axes[3].set_ylabel("TTMn rate"); axes[3].set_xlabel("game frame")
plt.tight_layout(); plt.show()

## 5. From a rate to a button

Neurons don't press buttons. An **action** plug-in (`actions/jump.py`) reads a group's rate and decides. *Jump* holds **A** while TTMn is above a threshold (and lets go at a lower one, so a flicker doesn't chop a jump).
The thresholds are `KnobSpec`s with `origin="invented"`: our choice. Let's find when the fly would press A during this approach.

In [ ]:
jump = registry.list_plugins("actions")["jump"]
knobs = {k.key: k.default for k in jump.knobs}
print(knobs)

state = {}
pressed = []
for t in range(len(gaps)):
    state["airborne"] = False
    intent = jump.decide({"TTMn": r["TTMn"][t]}, knobs, state)
    pressed.append("A" in intent.press)             # a ButtonIntent says which buttons to press / release
first = int(np.argmax(pressed)) if any(pressed) else None
print("A is first pressed at frame", first, "when the goomba is", None if first is None else round(float(gaps[first]), 1), "px away")

The jump comes late: the TTMn rate only crosses the threshold when the goomba is already very close, which is realistic for an *escape* reflex — but also why the thresholds are among the dials the evolution in the background tunes.

## 6. What is doing the work? Two experiments

**(a) Which detector matters?** Re-run the approach driving only LPLC2, only LC4, or both.

In [ ]:
def peak_ttmn(inputs_at):
    return float(simulate(inputs_at, len(gaps))["TTMn"].max())

both = peak_ttmn(lambda t: {"LPLC2_R": theta[t], "LC4_R": dtheta[t]})
only_lplc2 = peak_ttmn(lambda t: {"LPLC2_R": theta[t]})
only_lc4 = peak_ttmn(lambda t: {"LC4_R": dtheta[t]})
print(f"peak TTMn  both: {both:.3f} | LPLC2 only: {only_lplc2:.3f} | LC4 only: {only_lc4:.3f}   (jump threshold: {knobs['jump.ttmn_thresh']})")

With *our* input scaling, LPLC2 supplies almost all of the drive and LC4 alone stays far below the jump threshold. Careful: that depends on the invented cue (|Δθ| is a small number compared with θ), not only on the wiring. Change the cue's scaling and the balance changes: another reminder of which parts are orange.

**(b) What does the recurrence do?** The detector pools are densely connected to themselves (notebook 02). Make a copy of the brain with that internal wiring **removed** (set the detector→detector block of **W** to zero) and compare.
`Brain(spec, W, body_ids, types, sides, signs)` is the same public constructor the loader uses.

In [ ]:
masks = brain.masks
detectors = masks["LC4_L"] | masks["LC4_R"] | masks["LPLC2_L"] | masks["LPLC2_R"]
W_cut = brain.W.copy()
W_cut[np.ix_(detectors, detectors)] = 0.0            # no detector -> detector connections
brain_cut = fb.Brain(brain.spec, W_cut, brain.body_ids, brain.types, brain.sides, brain.signs)

inputs_at = lambda t: {"LPLC2_R": theta[t], "LC4_R": dtheta[t]}
normal = simulate(inputs_at, len(gaps))
cut = simulate(inputs_at, len(gaps), brain=brain_cut)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(normal["TTMn"], label="real wiring"); ax.plot(cut["TTMn"], label="detector-to-detector wiring removed")
ax.axhline(knobs["jump.ttmn_thresh"], color="grey", ls="--", label="jump threshold")
ax.set_xlabel("game frame"); ax.set_ylabel("TTMn rate"); ax.legend(); plt.show()
print(f"peak TTMn: real {normal['TTMn'].max():.3f} vs no recurrence {cut['TTMn'].max():.3f}  ({cut['TTMn'].max() / normal['TTMn'].max():.0%})")

Without the lateral wiring the response is clearly weaker: the recurrence **amplifies** the input. This is the reason (see the docstring of `fly/mechanisms/escape.py`) the circuit is run as a dynamical system and never collapsed into a weighted sum.

## 7. Save and restore the brain's state (rewind!)

For *rewind and retry* the tool must also save the brain. `brain.get_state(h)` returns the rates **keyed by body id** (not by row), so a state survives a rebuilt brain; `set_state` loads it.

In [ ]:
h = brain.new_state()
for t in range(40):
    h = brain.advance(h, brain.drive(inputs_at(t)), model)
saved = brain.get_state(h)                                  # {body id: rate}
print("saved", len(saved), "neurons; first three:", list(saved.items())[:3])

def finish(h0):
    h = h0.copy()
    for t in range(40, len(gaps)):
        h = brain.advance(h, brain.drive(inputs_at(t)), model)
    return brain.rates(h)["TTMn"]

a = finish(h)
b = finish(brain.set_state(saved))                          # rewind, then replay the rest
print("TTMn at the end: straight through =", round(a, 6), "| after rewind =", round(b, 6), "| identical:", a == b)

## Recap

- The brain is **W** (teal, from the connectome) plus a small update rule (orange, invented). Input is a number per group, output is a rate per group.
- The jump comes from a chain: looming detectors → Giant Fiber → TTMn → *threshold* → button **A**.
- The response is roughly proportional to the input; the **recurrence** amplifies it; the **threshold** creates the all-or-nothing behaviour.
- State is saved by body id, so the tool can rewind both the game and the brain.

**Try this**
1. Drive `LPLC2_L` instead of `LPLC2_R`. Does the Giant Fiber care which eye?
2. Change `model.g` (`fb.RateModel(g=...)`) and rerun section 4. When does the response saturate?
3. Remove only the LC4 recurrence (instead of all detectors) in section 6b.

**Next:** notebook 07 joins this brain to the game and runs one whole episode.